In [12]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [13]:
from sqlalchemy import create_engine
from langchain_community.utilities import SQLDatabase

engine = create_engine("sqlite:///inventory.db")

with engine.begin() as conn:

    conn.exec_driver_sql("""
    CREATE TABLE inventory (
        sku TEXT PRIMARY KEY,
        product_name TEXT,
        state TEXT,
        stock_count INTEGER,
        price REAL
    )
    """)

    mock_data = [
        ("SKU-992", "Wireless Mouse", "California", 150, 29.99),
        ("SKU-993", "Mechanical Keyboard", "California", 12, 89.99),
        ("SKU-994", "UltraWide Monitor", "Texas", 45, 349.99),
        ("SKU-995", "USB-C Hub", "California", 400, 19.99),
        ("SKU-996", "Ergonomic Desk Chair", "Texas", 5, 249.99)
    ]

    for row in mock_data:
        conn.exec_driver_sql(
            "INSERT INTO inventory VALUES (?, ?, ?, ?, ?)",
            row
        )

db = SQLDatabase(engine)
db

In [18]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda, RunnableParallel
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

In [15]:
db.get_table_info()

'\nCREATE TABLE inventory (\n\tsku TEXT, \n\tproduct_name TEXT, \n\tstate TEXT, \n\tstock_count INTEGER, \n\tprice REAL, \n\tPRIMARY KEY (sku)\n)\n\n/*\n3 rows from inventory table:\nsku\tproduct_name\tstate\tstock_count\tprice\nSKU-992\tWireless Mouse\tCalifornia\t150\t29.99\nSKU-993\tMechanical Keyboard\tCalifornia\t12\t89.99\nSKU-994\tUltraWide Monitor\tTexas\t45\t349.99\n*/'

In [16]:
def get_schema(_):
    return db.get_table_info()

# cleans the ouput given by the LLM
def run_query(sql_query: str):
    cleaned_query = (
        sql_query
        .strip()
        .replace("```sql", "")
        .replace("```", "")
        .strip()
    )

    return db.run(cleaned_query)

In [ ]:
sql_generation_prompt = ChatPromptTemplate.from_template(
"""
You are a SQLite expert.

Given the database schema below, write a SQL query that answers the user's question.

Return ONLY the SQL query.
Do not include explanations.
Do not use markdown.

Schema:
{schema}

Question:
{question}

SQL Query:
"""
)

sql_generation_chain = (
    RunnableParallel({
        "schema": RunnableLambda(get_schema),
        "question": RunnablePassthrough()
    })
    | sql_generation_prompt
    | llm
    | StrOutputParser()
)

sql_generation_chain.invoke("What are the top 3 most expensive products in California?")

"SELECT product_name, price FROM inventory WHERE state = 'California' ORDER BY price DESC LIMIT 3"

In [20]:
response_prompt = ChatPromptTemplate.from_template(
"""
You are a helpful business intelligence assistant.

Question:
{question}

SQL Query:
{query}

SQL Result:
{result}

Provide a concise natural language answer.
"""
)

full_text_to_sql_chain = (
    RunnableParallel({
        "question": RunnablePassthrough(),
        "query": sql_generation_chain,
    })
    | RunnableLambda(
        lambda x: {
            "question": x["question"],
            "query": x["query"],
            "result": run_query(x["query"])
        }
    )
    | response_prompt
    | llm
    | StrOutputParser()
)

In [22]:
print(db.run("SELECT * FROM inventory"))

[('SKU-992', 'Wireless Mouse', 'California', 150, 29.99), ('SKU-993', 'Mechanical Keyboard', 'California', 12, 89.99), ('SKU-994', 'UltraWide Monitor', 'Texas', 45, 349.99), ('SKU-995', 'USB-C Hub', 'California', 400, 19.99), ('SKU-996', 'Ergonomic Desk Chair', 'Texas', 5, 249.99)]


In [21]:
full_text_to_sql_chain.invoke(
    "What are the top 3 most expensive products in California?"
)

'The top 3 most expensive products in California are: Mechanical Keyboard ($89.99), Wireless Mouse ($29.99), and USB-C Hub ($19.99).'